## Prueba 

In [ ]:
#instalar librerias  

#! pip install pandas numpy faker openpyxl matplotlib seaborn jupyter

In [2]:
#importar librerias 

import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta
import os

In [3]:

# Configuración
fake = Faker('es_ES')  # Español de España
np.random.seed(42)     # Para reproducibilidad
Faker.seed(42)

In [4]:
 # Creacion de carpeta datos 
os.makedirs('../datos', exist_ok=True)

print("=" * 60)
print("🏗️  GENERADOR DE DATOS SINTÉTICOS - TFM COBROS PYMES")
print("=" * 60)

🏗️  GENERADOR DE DATOS SINTÉTICOS - TFM COBROS PYMES


In [5]:
# TABLA 1: CLIENTES

print("\n📋 Generando tabla de CLIENTES...")

NUM_CLIENTES = 100
sectores = ['Construcción', 'Hostelería', 'Retail', 'Tecnología', 
            'Consultoría', 'Transporte', 'Alimentación']

clientes = []
for i in range(NUM_CLIENTES):
    sector = np.random.choice(sectores)
    
    # Scoring externo (correlacionado con sector)
    # Tecnología y Consultoría son más solventes
    if sector in ['Tecnología', 'Consultoría']:
        scoring = np.random.choice(['Alto', 'Medio', 'Bajo'], p=[0.6, 0.3, 0.1])
    elif sector == 'Construcción':
        scoring = np.random.choice(['Alto', 'Medio', 'Bajo'], p=[0.1, 0.4, 0.5])
    else:
        scoring = np.random.choice(['Alto', 'Medio', 'Bajo'], p=[0.2, 0.5, 0.3])
    
    clientes.append({
        'cliente_id': f'CLI_{i+1:03d}',
        'nombre': fake.company(),
        'cif': fake.vat_id().replace('ES', ''),  # Formato español
        'sector': sector,
        'scoring_externo': scoring,
        'provincia': fake.state()
    })

df_clientes = pd.DataFrame(clientes)

print(f"✅ Generados {len(df_clientes)} clientes")
print(f"   Distribución por sector:")
print(df_clientes['sector'].value_counts().to_string())


📋 Generando tabla de CLIENTES...
✅ Generados 100 clientes
   Distribución por sector:
sector
Alimentación    24
Tecnología      17
Construcción    16
Consultoría     13
Transporte      12
Hostelería       9
Retail           9


In [6]:
# TABLA 1: CLIENTES

print("\n📋 Generando tabla de CLIENTES...")

NUM_CLIENTES = 100
sectores = ['Construcción', 'Hostelería', 'Retail', 'Tecnología', 
            'Consultoría', 'Transporte', 'Alimentación']

clientes = []
for i in range(NUM_CLIENTES):
    sector = np.random.choice(sectores)
    
    # Scoring externo
    if sector in ['Tecnología', 'Consultoría']:
        scoring = np.random.choice(['Alto', 'Medio', 'Bajo'], p=[0.6, 0.3, 0.1])
    elif sector == 'Construcción':
        scoring = np.random.choice(['Alto', 'Medio', 'Bajo'], p=[0.1, 0.4, 0.5])
    else:
        scoring = np.random.choice(['Alto', 'Medio', 'Bajo'], p=[0.2, 0.5, 0.3])
    
    clientes.append({
        'cliente_id': f'CLI_{i+1:03d}',
        'nombre': fake.company(),
        'cif': fake.vat_id().replace('ES', ''), 
        'sector': sector,
        'scoring_externo': scoring,
        'provincia': fake.region() # Usamos .region() o .administrative_unit() si .state() falla
    })

df_clientes = pd.DataFrame(clientes)
print(f"✅ Generados {len(df_clientes)} clientes")


📋 Generando tabla de CLIENTES...
✅ Generados 100 clientes


In [7]:
# TABLA 2: HISTORIAL DE PAGOS (antes de generar facturas)

print("\n📊 Generando HISTORIAL DE PAGOS...")

historial = []
for _, cliente in df_clientes.iterrows():
    # Número de facturas previas (entre 5 y 30)
    num_facturas = np.random.randint(5, 31)
    
    # Tasa de pago a tiempo correlacionada con scoring
    if cliente['scoring_externo'] == 'Alto':
        tasa_pago = np.random.uniform(0.80, 0.95)
        avg_retraso = np.random.uniform(2, 8)
    elif cliente['scoring_externo'] == 'Medio':
        tasa_pago = np.random.uniform(0.60, 0.80)
        avg_retraso = np.random.uniform(8, 20)
    else:  # Bajo
        tasa_pago = np.random.uniform(0.30, 0.60)
        avg_retraso = np.random.uniform(20, 45)
    
    historial.append({
        'cliente_id': cliente['cliente_id'],
        'facturas_totales': num_facturas,
        'facturas_pagadas': int(num_facturas * tasa_pago),
        'tasa_pago_a_tiempo': round(tasa_pago, 2),
        'avg_dias_retraso': round(avg_retraso, 1),
        'importe_promedio_historico': round(np.random.uniform(500, 8000), 2)
    })

df_historial = pd.DataFrame(historial)

print(f"✅ Generado historial para {len(df_historial)} clientes")
print(f"   Tasa promedio de pago a tiempo: {df_historial['tasa_pago_a_tiempo'].mean():.1%}")



📊 Generando HISTORIAL DE PAGOS...
✅ Generado historial para 100 clientes
   Tasa promedio de pago a tiempo: 70.2%


In [8]:
# TABLA 3: FACTURAS (la más importante)
print("\n💰 Generando tabla de FACTURAS...")

NUM_FACTURAS = 500
fecha_hoy = datetime.now()

facturas = []
for i in range(NUM_FACTURAS):
    # Seleccionar cliente aleatorio
    cliente = df_clientes.sample(1).iloc[0]
    hist = df_historial[df_historial['cliente_id'] == cliente['cliente_id']].iloc[0]
    
    # Fecha de emisión (entre 1 y 180 días atrás)
    dias_atras = np.random.randint(1, 181)
    fecha_emision = fecha_hoy - timedelta(days=dias_atras)
    
    # Vencimiento (típicamente 30 días después de emisión)
    plazo_pago = int(np.random.choice([15, 30, 45, 60], p=[0.1, 0.7, 0.15, 0.05]))
    vencimiento = fecha_emision + timedelta(days=plazo_pago)
    
    # Calcular días de retraso
    dias_desde_vencimiento = (fecha_hoy - vencimiento).days
    dias_retraso = max(0, dias_desde_vencimiento)
    
    # Decidir si está pagada (basado en historial del cliente)
    # Si ya venció, usar la tasa histórica
    if dias_desde_vencimiento > 0:
        # Probabilidad de estar pagada disminuye con el retraso
        prob_pagada = hist['tasa_pago_a_tiempo'] * (1 - min(dias_retraso / 90, 0.5))
        esta_pagada = np.random.random() < prob_pagada
    else:
        # Si no ha vencido, está pendiente
        esta_pagada = False
    
    # Importe (variación alrededor del promedio histórico)
    importe_base = hist['importe_promedio_historico']
    importe = round(np.random.normal(importe_base, importe_base * 0.3), 2)
    importe = max(100, importe)  # Mínimo 100€
    
    # Estado
    if esta_pagada:
        estado = 'Pagada'
        dias_retraso_final = 0
    elif dias_retraso > 0:
        estado = 'Vencida'
        dias_retraso_final = dias_retraso
    else:
        estado = 'Pendiente'
        dias_retraso_final = 0
    
    facturas.append({
        'factura_id': f'FAC-2025-{i+1:04d}',
        'cliente_id': cliente['cliente_id'],
        'importe': importe,
        'fecha_emision': fecha_emision.strftime('%Y-%m-%d'),
        'vencimiento': vencimiento.strftime('%Y-%m-%d'),
        'dias_retraso': dias_retraso_final,
        'estado': estado,
        'pagada': esta_pagada
    })

df_facturas = pd.DataFrame(facturas)

print(f"✅ Generadas {len(df_facturas)} facturas")
print(f"\n📊 Distribución de estados:")
print(df_facturas['estado'].value_counts().to_string())
print(f"\n💸 Importe total pendiente: {df_facturas[~df_facturas['pagada']]['importe'].sum():,.2f}€")
print(f"📈 Tasa de morosidad (vencidas/total): {(df_facturas['estado'] == 'Vencida').mean():.1%}")


💰 Generando tabla de FACTURAS...
✅ Generadas 500 facturas

📊 Distribución de estados:
estado
Vencida      237
Pagada       173
Pendiente     90

💸 Importe total pendiente: 1,198,672.20€
📈 Tasa de morosidad (vencidas/total): 47.4%


In [9]:
# GUARDAR ARCHIVOS CSV
print("\n💾 Guardando archivos CSV...")

ruta_clientes = '../datos/clientes.csv'
ruta_facturas = '../datos/facturas.csv'
ruta_historial = '../datos/historial_pagos.csv'

df_clientes.to_csv(ruta_clientes, index=False, encoding='utf-8')
df_facturas.to_csv(ruta_facturas, index=False, encoding='utf-8')
df_historial.to_csv(ruta_historial, index=False, encoding='utf-8')

print(f"✅ {ruta_clientes}")
print(f"✅ {ruta_facturas}")
print(f"✅ {ruta_historial}")


💾 Guardando archivos CSV...
✅ ../datos/clientes.csv
✅ ../datos/facturas.csv
✅ ../datos/historial_pagos.csv


In [11]:
# ESTADÍSTICAS FINALES

print("\n" + "=" * 60)
print("📊 RESUMEN FINAL")
print("=" * 60)

print(f"\n🏢 CLIENTES:")
print(f"   Total: {len(df_clientes)}")
print(f"   Scoring Alto: {(df_clientes['scoring_externo'] == 'Alto').sum()}")
print(f"   Scoring Medio: {(df_clientes['scoring_externo'] == 'Medio').sum()}")
print(f"   Scoring Bajo: {(df_clientes['scoring_externo'] == 'Bajo').sum()}")

print(f"\n💰 FACTURAS:")
print(f"   Total: {len(df_facturas)}")
print(f"   Pagadas: {(df_facturas['pagada']).sum()} ({(df_facturas['pagada']).mean():.1%})")
print(f"   Pendientes: {(df_facturas['estado'] == 'Pendiente').sum()}")
print(f"   Vencidas: {(df_facturas['estado'] == 'Vencida').sum()}")
print(f"   Importe total: {df_facturas['importe'].sum():,.2f}€")
print(f"   Importe pendiente: {df_facturas[~df_facturas['pagada']]['importe'].sum():,.2f}€")

print(f"\n⏱️  DÍAS DE RETRASO:")
facturas_vencidas = df_facturas[df_facturas['estado'] == 'Vencida']
if len(facturas_vencidas) > 0:
    print(f"   Promedio: {facturas_vencidas['dias_retraso'].mean():.1f} días")
    print(f"   Máximo: {facturas_vencidas['dias_retraso'].max():.0f} días")
    print(f"   Mediana: {facturas_vencidas['dias_retraso'].median():.1f} días")

print("\n✅ Generación completada exitosamente")
print("=" * 60)


📊 RESUMEN FINAL

🏢 CLIENTES:
   Total: 100
   Scoring Alto: 32
   Scoring Medio: 43
   Scoring Bajo: 25

💰 FACTURAS:
   Total: 500
   Pagadas: 173 (34.6%)
   Pendientes: 90
   Vencidas: 237
   Importe total: 1,836,441.18€
   Importe pendiente: 1,198,672.20€

⏱️  DÍAS DE RETRASO:
   Promedio: 78.8 días
   Máximo: 159 días
   Mediana: 81.0 días

✅ Generación completada exitosamente
